TASK 1 · Load and inspect

In [52]:
import pandas as pd
df=pd.read_csv('bangalore_tech_salaries.csv')
null=df.isnull().sum()
print(df.head()) #Employee_ID ,Role,years_exp,Current_CTC,Previous_CTC,Company,ompany_TYPE ,skills,Location ,Education_Tier ,Joining_Year,Work_Mode
# print(df.info())
# print(null)
print(df.describe())

  Employee_ID         Role   years_exp Current_CTC Previous_CTC       Company  \
0     BLR0065            DS          6        49.4     32.4 LPA           Ola   
1     BLR0080        DevOps          0         9.7          NaN  Walmart Labs   
2     BLR0166   SDE Backend          6   3,360,000     24.2 LPA       Postman   
3     BLR0219  SDE Frontend          8    41.4 LPA         30.3  Amazon India   
4     BLR0491  Product Lead          5   3,640,000     30.6 LPA   QuickPay AI   

  company_TYPE                                             Skills  \
0      Unicorn  JIRA, JavaScript, Spring Boot, Figma, NumPy, A...   
1          MNC     Tableau, Python, Figma, JavaScript, TensorFlow   
2     Mid-size               Agile, Excel, GCP, MongoDB, Java, ML   
3          MNC                      PyTorch, Redis, System Design   
4  early-stage                      Tableau, Deep Learning, Agile   

          Location Education_Tier  Joining_Year         Work_Mode  
0       HSR Layout         Tie

TASK 2 · Clean

In [92]:
df = df.rename(columns={'Employee_ID': 'eid', 'Role ': 'role'})#the  Employee_ID' column has changed to eid
colunmsname=df.columns
print(colunmsname)

# df.drop_duplicates(inplace=True)
# print(df.duplicated())
print("Current columns:", df.columns.tolist())

#Standardise role variants to canonical name
df['role'] = df['role'].astype(str).str.strip().str.title()
print(df['role'].unique())
role_mapping={
    "Ds":"Data Scientist",
    "Devops":"Devops Engineer",
    "Sde Backend":"Backend Developer",
    "Sde Frontend":"Frontend Developer",
    "Sde Fullstack":"Fullstack Developer",
    "Sre":"Site Reliability Engineer",
    "Designer":"UI/UX Designer",
    "Sr Designer":"UI/UX Designer",
    "Sr Pm":"promnt manger",
    "Be":"base enginer"

}
role_education={
    "2":"Tier-2",
    "3":"Tier-3",
    "T2":"Tier-2",
    "T3":"Tier-3",
    "1":"Tier-1",
    "T1":"Tier-1"

}
df['Education_Tier']=df['Education_Tier'].replace(role_education)
df['role']=df['role'].replace(role_mapping)
print(df['role'].unique())
import numpy as np
#converting current_ctc form str to float and heanding four case like 2.3 and 2.3lpa and 230000 ,$2.3lpa
def clean(val):
  if pd.isna(val):
    return np.nan
  val_str=str(val).upper().replace('LPA','').replace('₹','').replace(',','').strip()
  try:
    num=float(val_str)
    if num>1000:
      num/=100000
    return num
  except ValueError:
    return np.nan
df['Current_CTC']=df['Current_CTC'].apply(clean)

df['Previous_CTC']=df['Previous_CTC'].apply(clean)
#drop the duplicates
df=df.drop_duplicates().reset_index(drop=True)
if 'Education_Tier' in df.columns:
    df['Education_Tier_clean'] = (
        df['Education_Tier']
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(' ', '-')
        .str.replace('TIER', 'Tier') )
if 'Company_Type' in df.columns:
    company_map = lambda x: (
        'Product' if 'product' in str(x).lower()
        else ('Service' if 'service' in str(x).lower() else 'Startup')
    )

    df['Company_Type_clean'] = df['Company_Type'].apply(company_map)
print(df['role'].value_counts(dropna=False))
if 'Company_Type_clean' in df.columns:
    print("\n=== COMPANY TYPE VALUE COUNTS ===")
    print(df['Company_Type_clean'].value_counts(dropna=False))

if 'Education_Tier_clean' in df.columns:
    print("\n=== EDUCATION TIER VALUE COUNTS ===")
    print(df['Education_Tier_clean'].value_counts(dropna=False))

Index(['eid', 'role', 'years_exp', 'Current_CTC', 'Previous_CTC', 'Company',
       'company_TYPE', 'Skills', 'Location', 'Education_Tier', 'Joining_Year',
       'Work_Mode', 'Education_Tier_clean'],
      dtype='object')
Current columns: ['eid', 'role', 'years_exp', 'Current_CTC', 'Previous_CTC', 'Company', 'company_TYPE', 'Skills', 'Location', 'Education_Tier', 'Joining_Year', 'Work_Mode', 'Education_Tier_clean']
['Nan']
['Nan']
role
Nan    1000
Name: count, dtype: int64

=== EDUCATION TIER VALUE COUNTS ===
Education_Tier_clean
Tier-2    500
Tier-3    306
Tier-1    194
Name: count, dtype: int64


TASK 3 · Five business questions

Q3.1 — CTC distribution by role. For every role, compute median, mean, min, and max CTC. Sort roles by
median descending. Which role pays the most? Which the least? Is the mean materially different from
the median in any role (sign of outliers)?

In [96]:

unique_roles = df['role'].dropna().unique()
# 2. Process each role one by one using a loop
for role_name in unique_roles:
    # current CTCs
    role_salaries = df[df['role'] == role_name]['Current_CTC'].dropna().tolist()

    if len(role_salaries) == 0:
        continue

    # Calculate mean and median max min
    median_val = np.median(role_salaries)
    mean_val = np.mean(role_salaries)
    min_val = np.min(role_salaries)
    max_val = np.max(role_salaries)
    count_val = len(role_salaries)

    # Calculate the absolute difference between mean and median
    mean_median_diff = abs(mean_val - median_val)


    print("Count:", count_val)
    print("Median:", round(median_val, 2))
    print("Mean:", round(mean_val, 2))
    print("Min:", round(min_val, 2))
    print("Max:", round(max_val, 2))
    print("Mean-Median Difference:", round(mean_median_diff, 2))


Count: 1000
Median: 21.8
Mean: 24.17
Min: 5.2
Max: 84.4
Mean-Median Difference: 2.37


Q3.2 — Experience curve for SDE Backend. Bin SDE Backend professionals by years_exp into 4 buckets
(0-1, 2-3, 4-5, 6+). Compute median CTC per bucket. What is the CTC growth per experience band? Hint:
pd.cut + groupby.

In [99]:
backend_sde = df[df['role'] == 'Backend Developer'].copy()
bin_edges = [-1, 1, 3, 5, np.inf]
bin_labels = ['0-1', '2-3', '4-5', '6+']
backend_sde['exp_bin'] = pd.cut(backend_sde['years_exp'], bins=bin_edges, labels=bin_labels, right=False)
exp_curve = backend_sde.groupby('exp_bin',observed=False)['Current_CTC'].agg(['median','count'])
exp_curve['growth'] = exp_curve['median'].diff()
print(exp_curve)

         median  count  growth
exp_bin                       
0-1         NaN      0     NaN
2-3         NaN      0     NaN
4-5         NaN      0     NaN
6+          NaN      0     NaN
